In [ ]:
# 1. Install required packages (100% Offline Mode)
import os
import sys
import subprocess
import glob

os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

offline_pkg_dir = "/kaggle/input/datasets/mduy2911/offline-packages"
print(f"Installing offline packages from: {offline_pkg_dir}...")
wheels = glob.glob(os.path.join(offline_pkg_dir, "*.whl"))

if wheels:
    cmd = [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps"] + wheels
    subprocess.run(cmd, check=True)
    print("Offline installation completed successfully!")
else:
    raise FileNotFoundError(f"No offline wheels found in {offline_pkg_dir}!")

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
# 2. TRAINING HYPERPARAMETERS (Optimized for RTX Pro 6000 Ada / 96GB VRAM)
import os

MODEL_ID = "/kaggle/input/datasets/mduy2911/model-cache"

# --- RTX Pro 6000 Hardware Optimizations ---
USE_QLORA = False
GRADIENT_CHECKPOINTING = True

# --- Training Hyperparameters ---
MAX_LENGTH = 4096            # Maximum sequence length (optimized to cover Physics prompt and response)
BATCH_SIZE = 8            # Physical batch size optimized for RTX 6000
GRADIENT_ACCUMULATION = 4  # Gradient accumulation steps (Effective batch size = 32)
EPOCHS = 2                  # Number of training epochs
LEARNING_RATE = 1e-4        # Learning rate
OUTPUT_DIR = "/kaggle/working/results"  # Output directory on Kaggle

os.environ["WANDB_MODE"] = "disabled"
USE_WANDB = False


In [ ]:
# 3. Load NL -> FOL translation datasets, Physics dataset, and Physics Prompt
import os
import json

merged_path = "/kaggle/input/datasets/mduy2911/folc-train/logic_merged_valid_augmented.json"
physics_path = "/kaggle/input/datasets/mduy2911/folc-train/physics_distillation.json"

def load_translation_dataset(path):
    samples = []
    seen_premises = set()
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    for item in data:
        nl_list = item.get("premises-NL", [])
        fol_list = item.get("premises-FOL", [])
        if not nl_list or not fol_list or len(nl_list) != len(fol_list):
            continue
        nl_serialized = "\n".join(nl_list)
        if nl_serialized in seen_premises:
            continue
        seen_premises.add(nl_serialized)
        samples.append({
            "premises-NL": nl_list, 
            "premises-FOL": fol_list,
            "example_id": item.get("example_id", ""),
            "dataset_source": item.get("dataset_source", "")
        })
    print(f"Loaded {len(samples)} unique translation samples from {os.path.basename(path)}")
    return samples

def load_physics_dataset(path):
    samples = []
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    for item in data:
        inp = item.get("input", "")
        out = item.get("output", "")
        if inp and out:
            samples.append({"input": inp, "output": out})
    print(f"Loaded {len(samples)} physics samples from {os.path.basename(path)}")
    return samples

raw_samples = load_translation_dataset(merged_path)
physics_samples = load_physics_dataset(physics_path)

physics_system_prompt = """You are a precise physics reasoning agent.

TASK:
Convert a physics problem and any provided reasoning policies into executable SymPy code and a valid JSON response.

<REASONING_POLICY_OVERRIDE>

A <reasoning_policies> block may be provided.

If present:

1. Treat it as the primary reasoning guidance.

2. Follow its definitions, representations, topology rules,
   coordinate conventions, state models, and solution procedures.

3. Apply the underlying reasoning pattern rather than copying
   example expressions verbatim.

4. Override a policy only when required for:
   - physical validity
   - mathematical validity
   - SymPy executability

</REASONING_POLICY_OVERRIDE>

<OPERATING_CONSTRAINTS>

Return ONLY:

{
  "thought": "...",
  "physics_analysis": [...],
  "algebraic_reasoning": [...],
  "python_code": "...",
  "json_terminated": true
}

thought format:

<detected structure>. <activated reasoning pattern>. <solution strategy>.

thought:
- concise
- high-level
- no calculations
- no intermediate values
- no final values

physics_analysis:
- concise policy-grounded physical interpretation
- record relevant physical facts, assumptions, states, or constraints
- no calculations
- no final values

algebraic_reasoning:
- concise policy-grounded symbolic procedure
- describe the intended solve workflow
- no calculations
- no final values

python_code:
- begin with "import sympy as sp"
- use sp.Float(...) for numerical constants
- define variables before use
- solve symbolically before numeric evaluation
- evaluate norms, distances, and square roots using float(...) or .evalf()
- single-line string only
- separate statements using "; "
- no loops
- no functions
- no conditional branches
- no newline characters

Final code statements must define:

ans = [...]
unit = [...]

Requirements:
- ans must be a list
- unit must be a list
- len(ans) == len(unit)
- use raw SymPy-computed values
- do not manually round or format values
- no trailing semicolon after the final statement

Use SI base or SI derived units only.
Do not use engineering-prefix units.

Return the JSON object only.

</OPERATING_CONSTRAINTS>"""


In [ ]:
# 4. Initialize Tokenizer and Load Helper Functions
import os
import sys
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, PeftModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Check hardware bfloat16 support
if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    compute_dtype = torch.bfloat16
    use_bf16 = True
    use_fp16 = False
    print("GPU supports bfloat16. Using bfloat16 compute (Optimal for Ampere/Ada/Hopper GPUs like RTX Pro 6000).")
else:
    compute_dtype = torch.float16
    use_bf16 = False
    use_fp16 = True
    print("Using float16 compute (Standard for Turing/Pascal/Volta GPUs or CPU).")

# Select the most optimal Attention implementation
attn_impl = "sdpa"
try:
    import flash_attn
    attn_impl = "flash_attention_2"
    print("FlashAttention-2 is installed. Using flash_attention_2.")
except ImportError:
    print("FlashAttention-2 not found. Using PyTorch Native SDPA (Scaled Dot Product Attention).")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, 
    trust_remote_code=True, 
    local_files_only=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def load_base_model():
    print("Loading a fresh instance of the base model...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=compute_dtype if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
        trust_remote_code=True,
        attn_implementation=attn_impl,
        local_files_only=True
    )
    model.config.use_cache = False
    return model

def clean_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("GPU and system memory cleaned.")

target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# Optimized LoRA Configuration (Rank = 32, Alpha = 64) for logical capacity
peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=target_modules,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)


In [ ]:
# 5. Format Dataset (Chat Template) and split Train/Val
import os
import json
import random
from datasets import Dataset

# Define prompt templates for flat JSON list output with strict count constraint
system_prompt_template = (
    "You convert natural-language premises into parser-safe first-order logic formulas.\n\n"
    "Output a STRICT valid JSON list of strings containing the first-order logic formulas in the exact order of the input premises.\n"
    "You must output EXACTLY the same number of formulas as the input premises. Do not skip any premises or merge them.\n\n"
    "ALLOWED OPERATORS:\n"
    "AND, OR, NOT, ->, <->, =, !=, >=, <=, >, <, ForAll, Exists\n\n"
    "QUANTIFIER RULES:\n"
    "Use nested quantifiers only. E.g., ForAll(x, ForAll(y, P(x,y)))\n\n"
    "Return JSON only."
)

user_prompt_template = (
    "Convert the following {num_premises} premises into canonical first-order logic.\n\n"
    "Premises:\n"
    "{premises}\n\n"
    "Return a JSON list of exactly {num_premises} strings containing the formulas, in the exact same order."
)

# --- Split FOL dataset before augmentation to prevent data leakage ---
original_fol = []
augmented_fol = []
for sample in raw_samples:
    ds = str(sample.get("dataset_source", ""))
    if "augmented" in ds:
        augmented_fol.append(sample)
    else:
        original_fol.append(sample)

# Shuffle original FOL samples deterministically
random.Random(42).shuffle(original_fol)
split_idx_fol = int(len(original_fol) * 0.9)
train_orig_fol = original_fol[:split_idx_fol]
val_orig_fol = original_fol[split_idx_fol:]

# Map augmented samples back to train split
train_orig_ids = set(x["example_id"] for x in train_orig_fol)

def get_original_id(example_id):
    for suffix in ["_aug_var", "_perm_var", "_neg_var"]:
        if suffix in example_id:
            return example_id.split(suffix)[0]
    return example_id

train_augmented_fol = []
for sample in augmented_fol:
    base_id = get_original_id(sample["example_id"])
    if base_id in train_orig_ids:
        train_augmented_fol.append(sample)

# Combine splits
train_fol = train_orig_fol + train_augmented_fol
val_fol = val_orig_fol

# --- Split Physics dataset deterministically ---
random.Random(42).shuffle(physics_samples)
split_idx_phys = int(len(physics_samples) * 0.9)
train_physics = physics_samples[:split_idx_phys]
val_physics = physics_samples[split_idx_phys:]

# --- Format training and validation samples ---
def format_samples(fol_list, physics_list):
    formatted = []
    
    # Format FOL translation samples
    for item in fol_list:
        nl_list = item["premises-NL"]
        fol_list_item = item["premises-FOL"]
        
        nl_content = ""
        for i, nl in enumerate(nl_list, start=1):
            nl_content += f"{i}. {nl}\n"
            
        user_prompt = user_prompt_template.format(num_premises=len(nl_list), premises=nl_content.strip())
        assistant_response = json.dumps(fol_list_item, indent=2)
        
        formatted.append({
            "messages": [
                {"role": "system", "content": system_prompt_template},
                {"role": "user", "content": user_prompt.strip()},
                {"role": "assistant", "content": assistant_response.strip()}
            ]
        })
        
    # Format Physics samples
    for item in physics_list:
        physics_input = item["input"]
        physics_output = item["output"]
        
        formatted.append({
            "messages": [
                {"role": "system", "content": physics_system_prompt},
                {"role": "user", "content": physics_input.strip()},
                {"role": "assistant", "content": physics_output.strip()}
            ]
        })
        
    return formatted

formatted_train = format_samples(train_fol, train_physics)
formatted_val = format_samples(val_fol, val_physics)

train_dataset = Dataset.from_list(formatted_train)
val_dataset = Dataset.from_list(formatted_val)

def apply_template(batch):
    texts = []
    for messages in batch["messages"]:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return {"text": texts}

train_dataset = train_dataset.map(apply_template, batched=True, remove_columns=["messages"])
val_dataset = val_dataset.map(apply_template, batched=True, remove_columns=["messages"])

# Shuffle the final HF datasets
train_dataset = train_dataset.shuffle(seed=42)
val_dataset = val_dataset.shuffle(seed=42)

print(f"FOL Train/Val Split: original train={len(train_orig_fol)}, original val={len(val_orig_fol)}")
print(f"FOL Augmented added to Train: {len(train_augmented_fol)}")
print(f"Physics Train/Val Split: train={len(train_physics)}, val={len(val_physics)}")
print(f"Total Train size: {len(train_dataset)}, Total Val size: {len(val_dataset)}")


In [ ]:
# 6. Configure SFTTrainer and start training
import os
import glob
import torch
import json
import random
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback, DataCollatorForLanguageModeling
from typing import Dict, Union, Any, Optional, List, Tuple

class LossLoggingCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None:
            if "loss" in logs:
                print(f"Step {state.global_step}: training_loss = {logs['loss']:.6f}")
            if "eval_loss" in logs:
                print(f"Step {state.global_step}: validation_loss = {logs['eval_loss']:.6f}")

class CustomDataCollator(DataCollatorForLanguageModeling):
    def __init__(self, response_template, tokenizer, *args, **kwargs):
        super().__init__(tokenizer=tokenizer, mlm=False, *args, **kwargs)
        self.response_template = response_template
        self.response_token_ids = self.tokenizer.encode(self.response_template, add_special_tokens=False)
        
    def __call__(self, examples):
        batch = super().__call__(examples)
        labels = batch["labels"]
        for i in range(labels.size(0)):
            input_ids = batch["input_ids"][i].tolist()
            response_idx = -1
            n_template = len(self.response_token_ids)
            for j in range(len(input_ids) - n_template + 1):
                if input_ids[j:j+n_template] == self.response_token_ids:
                    response_idx = j + n_template
                    break
            
            if response_idx != -1:
                labels[i, :response_idx] = -100
                
        return batch

# --- ACCURACY EVALUATION HELPERS ---
def compare_physics_answers(pred_ans, pred_unit, gt_ans, gt_unit):
    if pred_ans is None or gt_ans is None:
        return False
    if not isinstance(pred_ans, list):
        pred_ans = [pred_ans]
    if not isinstance(gt_ans, list):
        gt_ans = [gt_ans]
    if len(pred_ans) != len(gt_ans):
        return False
    for p_val, g_val in zip(pred_ans, gt_ans):
        try:
            p_num = float(p_val)
            g_num = float(g_val)
            if g_num == 0.0:
                if abs(p_num) >= 1e-5:
                    return False
            else:
                rel_err = abs(p_num - g_num) / abs(g_num)
                if rel_err > 0.02:  # 2% tolerance
                    return False
        except (ValueError, TypeError):
            if str(p_val).strip().lower() != str(g_val).strip().lower():
                return False
    return True

def evaluate_fol_accuracy(model, tokenizer, val_samples, limit=50):
    print(f"Evaluating FOL Accuracy on {min(len(val_samples), limit)} samples...")
    correct_count = 0
    total_count = 0
    valid_json_count = 0
    
    # Shuffle and sample to keep evaluation fast
    random_subset = random.Random(42).sample(val_samples, min(len(val_samples), limit))
    
    for item in random_subset:
        nl_list = item["premises-NL"]
        fol_list_gt = item["premises-FOL"]
        
        nl_content = ""
        for i, nl in enumerate(nl_list, start=1):
            nl_content += f"{i}. {nl}\n"
            
        user_prompt = user_prompt_template.format(num_premises=len(nl_list), premises=nl_content.strip())
        messages = [
            {"role": "system", "content": system_prompt_template},
            {"role": "user", "content": user_prompt.strip()}
        ]
        
        prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
            
        generated_ids = outputs[0][inputs.input_ids.shape[1]:]
        response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
        
        try:
            parsed_response = json.loads(response)
            valid_json_count += 1
            if isinstance(parsed_response, list) and isinstance(fol_list_gt, list):
                if [str(x).strip() for x in parsed_response] == [str(x).strip() for x in fol_list_gt]:
                    correct_count += 1
        except Exception:
            pass
        total_count += 1
        
    em_acc = (correct_count / total_count) * 100 if total_count > 0 else 0
    json_rate = (valid_json_count / total_count) * 100 if total_count > 0 else 0
    print(f"FOL Exact Match Accuracy: {em_acc:.2f}% ({correct_count}/{total_count})")
    print(f"FOL Valid JSON Rate: {json_rate:.2f}% ({valid_json_count}/{total_count})")
    return em_acc, json_rate

def evaluate_physics_accuracy(model, tokenizer, val_samples, limit=50):
    print(f"Evaluating Physics Accuracy on {min(len(val_samples), limit)} samples...")
    correct_count = 0
    total_count = 0
    valid_json_count = 0
    exec_count = 0
    
    random_subset = random.Random(42).sample(val_samples, min(len(val_samples), limit))
    
    for item in random_subset:
        inp = item["input"]
        gt_output_str = item["output"]
        
        messages = [
            {"role": "system", "content": physics_system_prompt},
            {"role": "user", "content": inp.strip()}
        ]
        
        prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
            
        generated_ids = outputs[0][inputs.input_ids.shape[1]:]
        response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
        
        try:
            parsed_response = json.loads(response)
            valid_json_count += 1
            code_str = parsed_response.get("python_code", "")
            
            # Extract ground truth python code from output string
            try:
                gt_parsed = json.loads(gt_output_str)
                gt_code_str = gt_parsed.get("python_code", "")
            except Exception:
                gt_code_str = ""
                
            if code_str and gt_code_str:
                local_vars_pred = {}
                local_vars_gt = {}
                try:
                    import sympy as sp
                    # Execute predicted code
                    exec(code_str, {"sp": sp}, local_vars_pred)
                    pred_ans = local_vars_pred.get("ans", None)
                    pred_unit = local_vars_pred.get("unit", None)
                    
                    # Execute ground truth code
                    exec(gt_code_str, {"sp": sp}, local_vars_gt)
                    gt_ans = local_vars_gt.get("ans", None)
                    gt_unit = local_vars_gt.get("unit", None)
                    
                    if pred_ans is not None and gt_ans is not None:
                        exec_count += 1
                        if compare_physics_answers(pred_ans, pred_unit, gt_ans, gt_unit):
                            correct_count += 1
                except Exception:
                    pass
        except Exception:
            pass
        total_count += 1
        
    acc = (correct_count / total_count) * 100 if total_count > 0 else 0
    json_rate = (valid_json_count / total_count) * 100 if total_count > 0 else 0
    exec_rate = (exec_count / total_count) * 100 if total_count > 0 else 0
    print(f"Physics Accuracy: {acc:.2f}% ({correct_count}/{total_count})")
    print(f"Physics Valid JSON Rate: {json_rate:.2f}% ({valid_json_count}/{total_count})")
    print(f"Physics Code Execution Rate: {exec_rate:.2f}% ({exec_count}/{total_count})")
    return acc, json_rate, exec_rate

def train_model(train_dataset, val_dataset, output_dir):
    clean_memory()
    
    # Mathematically exact, dynamic warmup steps calculation based on actual dataset size
    num_samples = len(train_dataset)
    effective_batch_size = BATCH_SIZE * GRADIENT_ACCUMULATION
    steps_per_epoch = num_samples // effective_batch_size
    if num_samples % effective_batch_size != 0:
        steps_per_epoch += 1
    total_steps = steps_per_epoch * EPOCHS
    
    # Target exactly 5% warmup steps of total training steps (HF v5.2 compliant integer steps)
    warmup_steps = max(1, int(total_steps * 0.05))
    print(f"Training on {num_samples} samples. Steps per epoch: {steps_per_epoch}. Total steps: {total_steps}. Dynamic warmup steps: {warmup_steps}")
    
    base_model = load_base_model()
    
    # Resume from previous checkpoints if adapter_config.json exists in output_dir
    adapter_config_path = os.path.join(output_dir, "adapter_config.json")
    if os.path.exists(adapter_config_path):
        print(f"Resuming PEFT adapter weights from {output_dir}...")
        model = PeftModel.from_pretrained(base_model, output_dir, is_trainable=True)
    else:
        print("Initializing a new PEFT adapter...")
        model = get_peft_model(base_model, peft_config)
        
    model.print_trainable_parameters()
    
    # Highly Optimized SFTConfig with Cosine Decay Scheduler & Warmup
    sft_config = SFTConfig(
        output_dir=output_dir,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",             # Cosine learning rate decay scheduler
        warmup_steps=warmup_steps,              # Compliant, dynamic warm-up steps to stabilize updates
        fp16=use_fp16,
        bf16=use_bf16,
        logging_steps=1,
        logging_first_step=True,
        optim="adamw_torch",
        gradient_checkpointing=GRADIENT_CHECKPOINTING,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        per_device_eval_batch_size=BATCH_SIZE,
        report_to="none",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_total_limit=2,
        dataset_text_field="text",
        max_length=MAX_LENGTH,
        neftune_noise_alpha=None,
        dataloader_num_workers=2,
        dataloader_pin_memory=True
    )

    response_template = "<|im_start|>assistant\n"
    collator = CustomDataCollator(
        response_template=response_template, 
        tokenizer=tokenizer
    )
    
    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        args=sft_config,
        data_collator=collator,
        callbacks=[LossLoggingCallback()]
    )
    
    checkpoints = glob.glob(os.path.join(output_dir, "checkpoint-*"))
    resume_path = None
    if checkpoints:
        checkpoints.sort(key=lambda x: int(x.split("-")[-1]))
        resume_path = checkpoints[-1]
        print(f"Found checkpoints. Resuming training from: {resume_path}")
        
    trainer.train(resume_from_checkpoint=resume_path)
    
    print(f"Saving best validation adapter weights and tokenizer to {output_dir}...")
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    print("Training completed successfully!")
    
    # --- POST-TRAINING ACCURACY EVALUATION ---
    try:
        print("\n=== Starting Post-Training Accuracy Evaluation ===")
        model.eval()
        evaluate_fol_accuracy(model, tokenizer, val_fol, limit=50)
        evaluate_physics_accuracy(model, tokenizer, val_physics, limit=50)
    except Exception as e:
        print(f"Error during post-training evaluation: {e}")
    
    # Clean up model & trainer from memory
    del trainer
    del model
    del base_model
    clean_memory()
